In [ ]:
%pip install -q -U huggingface_hub datasets

In [20]:
from pathlib import Path
import sys
import os 
import time
import deeplake

os.environ["CUDA_VISIBLE_DEVICES"] = '0'

from PIL import Image
import torch
from torchvision import transforms

# uncomment for script
#REPO_ROOT = Path(__file__).resolve().parent 
REPO_ROOT = Path.cwd().resolve().parent
sys.path.append(str(REPO_ROOT / "Instant-GI"))

from generalizable_model.init_net import InitNet

CHECKPOINT = REPO_ROOT / "Instant-GI/checkpoints/epoch_best_ks_3_cupy.pth"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")
transform = transforms.Compose([
    transforms.ToTensor(),
])

def preprocess(path: Path) -> torch.Tensor:
    return transform(Image.open(path).convert("RGB"))

model = InitNet().to(device)
checkpoint = torch.load(CHECKPOINT, map_location=device)
model.load_state_dict(checkpoint["model"])
model.eval()

def load_image(image_path) -> torch.Tensor:
    image_tensor = preprocess(image_path).unsqueeze(0).to(device)
    return image_tensor

def infer(img_t: Path):
    image_tensor = img_t.unsqueeze(0).to(device)
    with torch.no_grad():
        xy, scaling, rotation, color, triangles = model(image_tensor, get_gaussians=True)
    return dict(
        xy=xy.squeeze(0),
        scaling=scaling.squeeze(0),
        rotation=rotation.squeeze(0),
        color=color.squeeze(0),
        triangles=triangles,  # triangles stay on CUDA unless you call .cpu()
    )



Using: cuda


In [ ]:
image_paths = [
    REPO_ROOT / f"Instant-GI/data/kodak/kodim{i:02d}.png" for i in range(1,24)
    # add more file paths to process sequentially
]

start_time = time.time()
for path in image_paths:
    img_tensor = load_image(path)
    gaussians = infer(img_tensor)
    print(f"{path.name}: {gaussians['xy'].shape[0]} Gaussians produced")

duration = time.time() - start_time

print(f"Total time {duration:0.2f}s")
print(f"Avg time {duration / len(image_paths):0.2f}s")


In [ ]:
# run 'hf auth login' and enter your token from huggingface

In [18]:
import torch
from datasets import load_dataset
from tqdm import tqdm
# No 'numpy' or 'PIL' needed here anymore

def process_imagenet_streaming():
    """
    Streams the ImageNet validation set using Hugging Face 'datasets'
    and gets PyTorch tensors directly.
    """
    print("\nLoading ImageNet-1k validation set from Hugging Face Hub (streaming)...")
    
    try:
        ds = load_dataset(
            "ILSVRC/imagenet-1k", 
            split="validation", 
            streaming=True, 
            #trust_remote_code=False
        )
    except Exception as e:
        print(f"Error loading dataset: {e}")
        print("Please ensure you are logged in via `huggingface-cli login` and have accepted the terms.")
        return

    results = {}
    
    # --- THIS IS THE KEY CHANGE ---
    # Tell the iterator to output PyTorch tensors instead of PIL images.
    ds_torch = ds.with_format("torch")
    # ------------------------------

    for i, sample in enumerate(tqdm(ds_torch, desc="Processing ImageNet (streaming)")):
        try:
            # 1. Get image (now a torch.Tensor)
            # It will have shape [3, H, W] and dtype uint8
            image_tensor_uint8 = sample['image']
            
            # 2. Preprocess the tensor
            # Handle grayscale images that load with 1 channel
            if image_tensor_uint8.shape[0] == 1:
                image_tensor_uint8 = image_tensor_uint8.repeat(3, 1, 1) # [1,H,W] -> [3,H,W]
                
            # Scale from uint8 [0, 255] to float [0.0, 1.0]
            image_tensor_float = image_tensor_uint8 / 255.0
            
            # 3. Run inference
            gaussians = infer(image_tensor_float)
            
            # Store a simple result
            results[i] = gaussians['xy'].shape[0]

        except Exception as e:
            print(f"Error processing image {i}: {e}")
            continue

    print(f"\nImageNet processing complete. Processed {len(results)} images.")
    if results:
        print(f"Sample result (Image 0): {results[0]} Gaussians produced")

In [21]:
process_imagenet_streaming()


Loading ImageNet-1k validation set from Hugging Face Hub (streaming)...


Processing ImageNet (streaming): 660it [04:29,  2.44it/s]


KeyboardInterrupt: 

# 2D Gaussian loss functs